# Sincronização permanente: Google Drive ↔ GitHub

Este notebook mantém uma cópia do **[vision-transfer-learning](https://github.com/Gengo250/vision-transfer-learning)** no **seu Google Drive**, não em uma pasta de trabalho temporária.

1. Execute a célula abaixo e autorize o acesso ao Google Drive.
2. O repositório ficará permanentemente em **Meu Drive/vision-transfer-learning** (caminho no Colab: `/content/drive/MyDrive/vision-transfer-learning`).
3. Na primeira execução, a célula faz `git clone`; nas próximas, `git pull --ff-only` na branch `main`.
4. Execute a célula final para verificar o commit e os arquivos disponíveis.

> `/content/drive` é somente o ponto de montagem do Google Drive. Os arquivos guardados em **MyDrive** permanecem na sua conta mesmo após o encerramento da sessão Colab.

> O `git pull` sincroniza **GitHub → Google Drive**. Mudanças locais no clone não são enviadas automaticamente ao GitHub. O notebook interrompe a atualização se detectar mudanças locais, evitando sobrescrevê-las. Para publicar mudanças de código feitas no clone, será necessário executar `git add/commit/push` com autenticação apropriada. A opção **Arquivo → Salvar uma cópia no GitHub** salva o notebook aberto no GitHub, mas não faz push de todos os arquivos modificados no clone.


In [ ]:
from pathlib import Path
import os
import subprocess
from google.colab import drive

# Monta o Google Drive: o conteudo em MyDrive e persistente.
drive.mount("/content/drive")

REPO_URL = "https://github.com/Gengo250/vision-transfer-learning.git"
BRANCH = "main"
REPO_DIR = Path("/content/drive/MyDrive/vision-transfer-learning")

def git(*args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe, mas nao e um repositorio Git.")

    current_branch = git("branch", "--show-current", cwd=REPO_DIR)
    if current_branch != BRANCH:
        raise RuntimeError(
            f"Branch atual: {current_branch}. Esperada: {BRANCH}. "
            "Troque de branch manualmente antes de atualizar."
        )

    changes = git("status", "--porcelain", cwd=REPO_DIR)
    if changes:
        raise RuntimeError(
            "Ha alteracoes locais no clone do Colab. "
            "Salve-as antes de executar git pull para evitar perda de trabalho.\n"
            + changes
        )

    print("Atualizando o repositorio...")
    print(git("pull", "--ff-only", "origin", BRANCH, cwd=REPO_DIR))
else:
    print("Clonando o repositorio...")
    print(git("clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)))

os.chdir(REPO_DIR)
print(f"\nDiretorio: {Path.cwd()}")
print("Commit:", git("log", "-1", "--format=%h %s", cwd=REPO_DIR))


## Conferir clone persistente

Verifique o caminho permanente, a branch e o estado do repositório.


In [ ]:
print("Branch:", git("branch", "--show-current", cwd=REPO_DIR))
print("Status:")
print(git("status", "--short", cwd=REPO_DIR) or "(sem alteracoes locais)")
print("\nArquivos na raiz:")
for path in sorted(REPO_DIR.iterdir()):
    if path.name != ".git":
        print("📁" if path.is_dir() else "📄", path.name)
